# LLM Information Extraction Workshop
## 1st pipeline:
**Two-Pass Discovery Method**

This notebook helps you extract structured information from text data using Large Language Models.

**The approach:**
1. **Pass 1 (Exploration):** Let the LLM freely discover patterns and categories in your data
2. **Pass 2 (Structured):** Use the discovered categories to do precise, consistent extraction

---

# 2nd pipeline:

Theme / keyword extraction - Summarization

## Step 0: Setup

Run this cell once to install and import everything needed.

In [ ]:
!apt-get install zstd

# Install Ollama (run once)
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import time
time.sleep(3)  # Wait for server to start

print("✓ Ollama installed and running!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,623 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

In [ ]:
# Install dependencies
!pip install dspy datasets pymupdf python-docx thefuzz pandas tqdm -q
print("✓ Dependencies installed!")

# Imports
import dspy
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from pydantic import BaseModel
from thefuzz import fuzz
from IPython.display import display, Markdown
import requests                     # For talking to Ollama
print("✓ Imports ready!")
print("✓ All imports successful!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.3/291.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 25.6 MB/s eta 0:00:00
✓ Dependencies installed!
✓ Imports ready!
✓ All imports successful!


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

In [ ]:
# Check GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available")

GPU: Tesla T4
Memory: 15.8 GB


---
## Step 1: Download a Model

Choose a model based on available hardware:

- **Small (4GB):** `gemma3:4b`
- **Medium (14GB):** `gpt-oss:20b`
- **Large (48GB+):** `llama3.1:70b`

Full list: https://ollama.com/library

In [ ]:
# ============================================================
# CHANGE THIS: Select your model
# ============================================================

MODEL_NAME = "gpt-oss:20b"  # <-- Change to your preferred model

# ============================================================

# Download the model (only needed once per model)
!ollama pull {MODEL_NAME}

print(f"\n✓ Model {MODEL_NAME} ready!")



✓ Model gpt-oss:20b ready!


In [ ]:
# Configure DSPy
lm = dspy.LM(
    f"ollama_chat/{MODEL_NAME}",
    api_base="http://localhost:11434",
    max_tokens=150  # Increase to avoid truncation
)
dspy.configure(lm=lm)
print("✓ DSPy configured!")

test = dspy.Predict("question -> answer")

✓ DSPy configured!


In [ ]:
response = test(question="Which of the following winter jackets brands gives the user more status, Moncler or Canada goose? Justification for your reasoning!!")
print(response.answer)

Moncler generally confers higher status than Canada Goose. While both brands are respected in the outerwear market, Moncler’s positioning as a luxury fashion house—evidenced by its runway presence, high price points (often exceeding $1,500 for a basic jacket), limited editions, and collaborations with high‑profile designers—creates an aura of exclusivity and prestige that extends beyond the functional appeal of the garment. Its heritage in alpine fashion and the iconic quilted design, often seen on celebrities and fashion influencers, reinforce its status as a status symbol.

Canada Goose, on the other hand, is celebrated for its durability, technical performance, and iconic “wings” logo. Its market positioning leans more toward functional outdoor wear and lifestyle branding. While it has earned a strong reputation and a certain “premium” image, its price range (typically $400–$800) and focus on rugged practicality tend to resonate more with adventurous consumers rather than fashion‑ce

In [ ]:
import subprocess
import time

# Kill any existing ollama process and restart
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("✓ Ollama server restarted!")

✓ Ollama server restarted!


---
## Step 2: Load Your Data

**Option A:** Load from text files in a folder  
**Option B:** Load from a ready-made dataset (CSV, HuggingFace, etc.)

In [ ]:
def load_huggingface_dataset(dataset_name, split, text_column, n_samples=100):
    """Load dataset from HuggingFace."""
    from datasets import load_dataset

    ds = load_dataset(dataset_name, split=f"{split}[:{n_samples}]")
    df = pd.DataFrame({
        "id": [f"{dataset_name}_{i}" for i in range(len(ds))],
        "text": ds[text_column]
    })
    print(f"✓ Loaded {len(df)} samples from {dataset_name}")
    return df

print("✓ HuggingFace loader ready!")

# Document chunking for PDFs/DOCX

DIVIDER = "\n\n--- CHUNK ---\n\n"


def extract_text_from_file(file_path):
    """Extract text from PDF, DOCX, or TXT."""
    path = Path(file_path)

    if path.suffix.lower() == ".pdf":
        import pymupdf
        doc = pymupdf.open(path)
        return "\n\n".join(page.get_text() for page in doc)
    elif path.suffix.lower() == ".docx":
        from docx import Document
        doc = Document(path)
        return "\n\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        return path.read_text(encoding="utf-8")


def auto_chunk_to_txt(input_file, output_file, target_words=3000):
    """Auto-chunk document to editable TXT file."""
    text = extract_text_from_file(input_file)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks, current_chunk, current_words = [], [], 0

    for para in paragraphs:
        para_words = len(para.split())
        if current_words + para_words > target_words and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            current_chunk, current_words = [para], para_words
        else:
            current_chunk.append(para)
            current_words += para_words

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    Path(output_file).write_text(DIVIDER.join(chunks), encoding="utf-8")

    print(f"✓ Created {len(chunks)} chunks → {output_file}")
    print(f"\n→ Edit the file, then run load_chunked_txt()")


def load_chunked_txt(file_path):
    """Load edited chunk file as DataFrame."""
    text = Path(file_path).read_text(encoding="utf-8")
    chunks = [c.strip() for c in text.split("--- CHUNK ---") if c.strip()]

    df = pd.DataFrame({
        "id": [f"chunk_{i:03d}" for i in range(len(chunks))],
        "text": chunks
    })
    print(f"✓ Loaded {len(df)} chunks")
    return df

print("✓ Chunking functions ready!")



✓ HuggingFace loader ready!
✓ Chunking functions ready!


In [ ]:
# Load IMDB dataset
df = load_huggingface_dataset("imdb", "train", "text", n_samples=25000).sample(n=1000)
# Or load your own:
# df = load_chunked_txt("my_chunks.txt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

✓ Loaded 25000 samples from imdb


In [ ]:
df.head()

,id,text
4707,imdb_4707,"The ""Confidential"" part was meant to piggy-bac..."
16879,imdb_16879,Don't be swayed by the naysayers. This is a wo...
1907,imdb_1907,"Antonioni, by making this film, had assumed th..."
1732,imdb_1732,"Man, what a scam this turned out to be! Not be..."
11332,imdb_11332,This movie will promote the improvement of the...


In [ ]:
# Preview first text
print(f"ID: {df.iloc[2]['id']}")
print(f"Length: {len(df.iloc[2]['text'])} chars")
print("="*50)
print(df.iloc[2]['text'][:1000] + "...")

ID: imdb_1907
Length: 4187 chars
Antonioni, by making this film, had assumed the role of Papa Smurf to all the little long-haired, American, radical student-Smurfs. He had taken them under the guiding protection of his European communist wings, showing appreciation and support for their confused American ways. (These Smurfs are red and wear blue, not the other way around.) The radical Smurfs were happy to get the guidance of a wise old man with gray hair who regularly preys to the God of all long-haired Smurfs, Lenin the Communist - another wise old man whose beard made the Smurfs take him even more seriously, for it symbolized something wise, though they did not quite know why they regarded the beard to have this kind of deep effect on them. Castro, another wise bearded man, has often profited from this confusion and exuded magical powers with his beard over his naive overseas admirers. (Not to mention Che Guevara: that beard has a certain je-ne-sais-pas-quoi about it, makes one want 

# Select your own data:

In [ ]:
# --- OPTION 2: Raw File -> Auto-Chunker ---
import os
from google.colab import files

INPUT_FILE = "/content/source_document.pdf" # Change extension as needed
CHUNKED_FILE = "chunks_review.txt"

if not os.path.exists(INPUT_FILE):
    print("Please upload your source file:")
    uploaded = files.upload()
    uploaded_filename = list(uploaded.keys())[0]
    Path(INPUT_FILE).write_bytes(uploaded[uploaded_filename])

# Process using your existing chunker
auto_chunk_to_txt(INPUT_FILE, CHUNKED_FILE, target_words=400)

# Load and take a random sample
df_themes = load_chunked_txt(CHUNKED_FILE)
df_themes = df_themes.sample(n=min(1000, len(df_themes))).reset_index(drop=True)

df_themes.head()

Please upload your source file:


Saving hayward_jeromes-hebrew-questions-on-genesis-translated-with-an-introduction-and-commentary-0198263503.pdf to hayward_jeromes-hebrew-questions-on-genesis-translated-with-an-introduction-and-commentary-0198263503.pdf
✓ Created 281 chunks → chunks_review.txt

→ Edit the file, then run load_chunked_txt()
✓ Loaded 281 chunks


,id,text
0,chunk_142,"Commentary\n135\n8: 6 as ‘door’, LXX created a..."
1,chunk_128,Commentary\n121\ndon or forgiveness. He probab...
2,chunk_189,"Commentary\n(PJ of Gen. 36: 32), Laban (b. San..."
3,chunk_099,"92-\nCommentary\nfrom Virgil’s Eclogue 6, line..."
4,chunk_198,Commentary\n191\nwords should be ‘translated’ ...


In [ ]:
# --- OPTION 3: Pre-Chunked File ---
CHUNKED_FILE = "chunks_review.txt"

if os.path.exists(CHUNKED_FILE):
    df_themes = load_chunked_txt(CHUNKED_FILE)
    df_themes = df_themes.sample(n=min(1000, len(df_themes))).reset_index(drop=True)
    print("Ready with existing chunks.")
else:
    print("No chunk file found. Run Option 2 first.")
df_themes.head()

✓ Loaded 281 chunks
Ready with existing chunks.


,id,text
0,chunk_016,Introduction\n9\nBut as Kamesar rightly object...
1,chunk_231,125) prefigured Christ’s passion and the purpl...
2,chunk_245,"Commentary\nthese Patriarchs as Pharisees, and..."
3,chunk_272,266\nIndex of Passages Cited\n1-777 \n155\n2 ....
4,chunk_145,i 3 8\nCommentary\nThus he could make the conv...


In [ ]:
#Data in correct order
df_themes = load_chunked_txt(CHUNKED_FILE).sort_values('id').head(1000)
df_themes.head()

✓ Loaded 281 chunks


,id,text
0,chunk_000,THE OXFORD EARLY CHRISTIAN STUDIES series \nwi...
1,chunk_001,"Oxford University Press, Walton Street, Oxford..."
2,chunk_002,"Magistris Meis \nAntonio Gelston, DD \net\nGez..."
3,chunk_003,PREFACE\nThis translation of Jerome’s Hebrew Q...
4,chunk_004,"viii\nPreface\nHalladay, Principal of St Chad’..."


# 1st Pipeline
## First pass extraction

In [ ]:
class Aspect(BaseModel):
    aspect: str      # Acting, plot, etc..
    sentiment: str   # positive, negative, neutral
    quote: str       # supporting quote - Reasoning

class AspectList(BaseModel):
    aspects: list[Aspect]

class ExtractAspects(dspy.Signature):
    """Extract aspects discussed in a review with sentiment and quotes."""
    text: str = dspy.InputField()
    aspects: AspectList = dspy.OutputField()

aspect_extractor = dspy.Predict(ExtractAspects)
print("✓ Extractor ready!")

✓ Extractor ready!


In [ ]:
result = aspect_extractor(text=df.iloc[2]['text'][:2500])

print("EXTRACTED ASPECTS:")
for asp in result.aspects.aspects:
    print(f"  [{asp.sentiment:8}] {asp.aspect}")
    print(f"            \"{asp.quote}...\"\n")

EXTRACTED ASPECTS:
  [critical] Antonioni
            "Papa Smurf to all the little long-haired, American, radical student-Smurfs...."

  [admiring] Lenin
            "Lenin the Communist - another wise old man whose beard......"

  [critical] Castro
            "Castro, another wise bearded man, has often profited from this confusion and exuded magical powers with his beard......"

  [positive] Che Guevara
            "that beard has a certain je-ne-sais-pas-quoi about it, makes one want to immediately embrace Marx and his lovely, pacifistic teachings..."

  [negative] Radical Students
            "a muddled meeting of radically stupid radical students, who engage in dialogues that truly redefine the word "confused". As confused as a blind-folded dog falling of a high-story building into a bottomless pit...."

  [negative] Cops
            "i.e. pigs at a rally......"

  [negative] Capitalism
            "Antonioni's predictable assault on capitalism is not only intellectually hollow.

In [ ]:
# ============================================================
# CHANGE THIS: Your extraction instructions
# ============================================================

ASPECT_INSTRUCTIONS = """
Extract all aspects the reviewer discusses (acting, plot, music, etc.).
For each: name the aspect, sentiment (positive/negative/neutral), and a quote.
"""

# ============================================================


# Preview prompt + data
df["prompt_preview"] = df["text"].apply(
    lambda x: f"{ASPECT_INSTRUCTIONS}\n\nTEXT TO PROCESS: \n \n{x[:2600]}..."
)

display(Markdown(df.iloc[2]["prompt_preview"]))


Extract all aspects the reviewer discusses (acting, plot, music, etc.).
For each: name the aspect, sentiment (positive/negative/neutral), and a quote.


TEXT TO PROCESS: 
 
Antonioni, by making this film, had assumed the role of Papa Smurf to all the little long-haired, American, radical student-Smurfs. He had taken them under the guiding protection of his European communist wings, showing appreciation and support for their confused American ways. (These Smurfs are red and wear blue, not the other way around.) The radical Smurfs were happy to get the guidance of a wise old man with gray hair who regularly preys to the God of all long-haired Smurfs, Lenin the Communist - another wise old man whose beard made the Smurfs take him even more seriously, for it symbolized something wise, though they did not quite know why they regarded the beard to have this kind of deep effect on them. Castro, another wise bearded man, has often profited from this confusion and exuded magical powers with his beard over his naive overseas admirers. (Not to mention Che Guevara: that beard has a certain je-ne-sais-pas-quoi about it, makes one want to immediately embrace Marx and his lovely, pacifistic teachings) The film starts with a muddled meeting of radically stupid radical students, who engage in dialogues that truly redefine the word "confused". As confused as a blind-folded dog falling of a high-story building into a bottomless pit. Suddenly, the movie's "hero" (well, Antonioni's hero) rises up and says something to his pathetic left-wing peers and then leaves, hoping that this display of "mega-coolness" will improve his James Dean image and vastly increase his chances of getting laid with the best "chicks" in the next mass hippie orgy. Eventually he gets into trouble with cops (i.e. pigs) at a rally, and spends the movie under the blue American capitalist skies, looking for freedom Or something like that.<br /><br />Antonioni's predictable assault on capitalism is not only intellectually hollow, but has (or had) nothing new to offer; it's just the same old trigger-happy one-dimensional cops, businessmen discussing business deals (and what's wrong with that, isn't that how Antonioni's movies get made?), and endless shots of TV commercials and billboards advertising the oh-so morally decadent products for the abhorrent, selfish, and greedy right-wing rabble-population who thinks of no one but themselves, their families, their work, and their children.<br /><br />Papa Smurf Antonioni, just like his long-haired Smurfs and Smurfettes of the late 60s, failed to notice the most obvious and vital aspect about their silly movement: they were allowed to have their laughable meetings and express their anti-establishment opinions freely within that ve...

In [ ]:
df[["id", "prompt_preview"]].head(5)

,id,prompt_preview
4707,imdb_4707,\nExtract all aspects the reviewer discusses (...
16879,imdb_16879,\nExtract all aspects the reviewer discusses (...
1907,imdb_1907,\nExtract all aspects the reviewer discusses (...
1732,imdb_1732,\nExtract all aspects the reviewer discusses (...
11332,imdb_11332,\nExtract all aspects the reviewer discusses (...


In [ ]:
# ============================================================
N_SAMPLES_PASS1 = 20  # Start small
# ============================================================

pass1_results = []
all_aspects = []

for idx, row in tqdm(df.head(N_SAMPLES_PASS1).iterrows(), total=N_SAMPLES_PASS1):
    try:
        result = aspect_extractor(text=row["text"][:2500])
        for asp in result.aspects.aspects:
            pass1_results.append({"id": row["id"], **asp.dict()})
            all_aspects.append(asp.aspect.lower())
    except Exception as e:
        print(f"Error {row['id']}: {e}")

print(f"\n✓ Found {len(pass1_results)} aspects")

  5%|▌         | 1/20 [00:17<05:41, 17.99s/it]

Error imdb_4707: Adapter JSONAdapter failed to parse the LM response. 

LM Response: {} 

Expected to find output fields in the LM response: [aspects] 

Actual output fields parsed from the LM response: [] 




/tmp/ipython-input-1995185975.py:12: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  pass1_results.append({"id": row["id"], **asp.dict()})
 20%|██        | 4/20 [00:46<02:56, 11.06s/it]

Error imdb_1732: 1 validation error for AspectList
  Input should be a valid dictionary or instance of AspectList [type=model_type, input_value=[{'aspect': 'DVD Sleeve M...sentiment': 'negative'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type


 30%|███       | 6/20 [01:08<02:34, 11.01s/it]

...

Error imdb_9337: Adapter JSONAdapter failed to parse the LM response. 

LM Response: {} 

Expected to find output fields in the LM response: [aspects] 

Actual output fields parsed from the LM response: [] 




 95%|█████████▌| 19/20 [03:54<00:11, 11.99s/it]

Error imdb_21430: Adapter JSONAdapter failed to parse the LM response. 

LM Response: {} 

Expected to find output fields in the LM response: [aspects] 

Actual output fields parsed from the LM response: [] 




100%|██████████| 20/20 [04:08<00:00, 12.41s/it]


✓ Found 54 aspects


In [ ]:
print("DISCOVERED ASPECTS:")
for aspect, count in Counter(all_aspects).most_common(100):
    print(f"  {count:3d}x  {aspect}")

pd.DataFrame(pass1_results).head(100)

DISCOVERED ASPECTS:
    2x  movie quality
    2x  comedy
    1x  film's spookiness
    1x  writer/director jt petty
    1x  plot mystery (child murder)
    1x  lack of dialogue
    1x  film's impact on senses
    1x  antonioni
    1x  lenin
    1x  castro
    1x  che guevara
    1x  radical students
    1x  cops
    1x  capitalism
    1x  film's hero
    1x  actor performance
    1x  embarrassment
    1x  recommendation
    1x  star trek
    1x  kirk's decision to grant crew time off
    1x  planet as a vacation destination
    1x  conflict on the planet
    1x  action and tense moments
    1x  space camp
    1x  jinx (robot)
    1x  space scenes
    1x  lea thompson
    1x  joaquin phoenix
    1x  romance
    1x  eerie sense
    1x  firestarter plot
    1x  special effects
    1x  visions of deceased parents
    1x  human experiment
    1x  film
    1x  christopher walken
    1x  jack black
    1x  ben stiller
    1x  amy poehler
    1x  wayans brothers
    1x  movie plot
    1x  comm

,id,aspect,sentiment,quote
0,imdb_16879,film's spookiness,positive,wonderfully spooky film
1,imdb_16879,writer/director JT Petty,positive,He did a great job of having me on the edge of...
2,imdb_16879,plot mystery (child murder),neutral,"he witnesses the murder of a child, or does he?"
3,imdb_16879,lack of dialogue,intriguing,no dialogue until the last few scenes
4,imdb_16879,film's impact on senses,positive,manages to get hold of your senses and gives t...
5,imdb_1907,Antonioni,critical,"Papa Smurf to all the little long-haired, Amer..."
6,imdb_1907,Lenin,admiring,Lenin the Communist - another wise old man who...
7,imdb_1907,Castro,critical,"Castro, another wise bearded man, has often pr..."
8,imdb_1907,Che Guevara,positive,that beard has a certain je-ne-sais-pas-quoi a...
9,imdb_1907,Radical Students,negative,a muddled meeting of radically stupid radical ...


# Second pass extraction with fixed categories

In [ ]:
# ============================================================
# CHANGE THIS: Consolidate similar aspects into categories
# ============================================================

FINAL_CATEGORIES = [
    "Acting",
    "Plot",
    "Directing",
    "Cinematography",
    "Music",
    "Dialogue",
    "Pacing",
    "Emotional Impact",
    "Comedy",
    "Recommendation",
    "Movie quality"
]

# ============================================================

print("Categories:", FINAL_CATEGORIES)



Categories: ['Acting', 'Plot', 'Directing', 'Cinematography', 'Music', 'Dialogue', 'Pacing', 'Emotional Impact', 'Comedy', 'Recommendation', 'Movie quality']


In [ ]:
# Build prompt with the categories list
categories_formatted = "\n".join(f"- {cat}" for cat in FINAL_CATEGORIES)


CUSTOM_PROMPT = f"""Extract aspects from a movie review.

ALLOWED CATEGORIES - Use ONLY these exact names:
{categories_formatted}

For each aspect found:
- category: MUST be one of the categories listed above (exactly as written)
- sentiment: positive, negative, or neutral (lowercase only)
- quote: short quote from the text supporting this aspect

If something doesn't fit the categories above, use "Other"."""

In [ ]:
class StructuredAspect(BaseModel):
    category: str
    sentiment: str
    quote: str


class ExtractStructured(dspy.Signature):
    __doc__ = CUSTOM_PROMPT

    text: str = dspy.InputField(desc="The review text to analyze")
    aspects: list[StructuredAspect] = dspy.OutputField(desc="List of extracted aspects")

structured_extractor = dspy.Predict(ExtractStructured)

# Verify prompt
print("PROMPT:")
print(CUSTOM_PROMPT)

PROMPT:
Extract aspects from a movie review.

ALLOWED CATEGORIES - Use ONLY these exact names:
- Acting
- Plot
- Directing
- Cinematography
- Music
- Dialogue
- Pacing
- Emotional Impact
- Comedy
- Recommendation
- Movie quality

For each aspect found:
- category: MUST be one of the categories listed above (exactly as written)
- sentiment: positive, negative, or neutral (lowercase only)
- quote: short quote from the text supporting this aspect

If something doesn't fit the categories above, use "Other".


In [ ]:
result = structured_extractor(text=df.iloc[0]['text'][:2500])

for asp in result.aspects:
    print(f"[{asp.sentiment:8}] {asp.category}: \"{asp.quote[:50]}...\"")

KeyboardInterrupt: 

In [ ]:
# After a call, inspect history
print(lm.history[-1])

In [ ]:
# ============================================================
N_SAMPLES_PASS2 = 15
# ============================================================

pass2_results = []

for idx, row in tqdm(df.head(N_SAMPLES_PASS2).iterrows(), total=N_SAMPLES_PASS2):
    try:
        result = structured_extractor(text=row["text"][:2500])
        for asp in result.aspects:
            pass2_results.append({
                "doc_id": row["id"],
                "category": asp.category,
                "sentiment": asp.sentiment,
                "quote": asp.quote
            })
    except Exception as e:
        print(f"Error {row['id']}: {e}")

print(f"\n✓ Extracted {len(pass2_results)} aspects")

results_df = pd.DataFrame(pass2_results)
results_df.head(10)

 60%|██████    | 9/15 [01:48<01:10, 11.77s/it]

Error imdb_2159: Adapter JSONAdapter failed to parse the LM response. 

LM Response: {} 

Expected to find output fields in the LM response: [aspects] 

Actual output fields parsed from the LM response: [] 




100%|██████████| 15/15 [03:06<00:00, 12.46s/it]


✓ Extracted 51 aspects


,doc_id,category,sentiment,quote
0,imdb_4707,Directing,negative,Salkow treats it as just another pay-day exerc...
1,imdb_4707,Acting,negative,Brian Keith's typical low-key style doesn't wo...
2,imdb_4707,Acting,negative,Elisha Cook Jr. goes over the top as a wild-ey...
3,imdb_4707,Movie quality,negative,a cheap-jack production all the way.
4,imdb_4707,Acting,positive,Beverly Garland who treats her role with chara...
5,imdb_16879,Emotional Impact,positive,gives them a relentless tug
6,imdb_16879,Plot,neutral,"witnesses the murder of a child, or does he?"
7,imdb_16879,Pacing,positive,here is no dialogue until the last few scenes
8,imdb_1907,Acting,negative,"this display of ""mega-coolness"""
9,imdb_1907,Plot,negative,"the movie's ""hero"" rises up and says something"


In [ ]:
results_df = pd.DataFrame(pass2_results)
results_df.head(50)



,doc_id,category,sentiment,quote
0,imdb_4707,Directing,negative,Salkow treats it as just another pay-day exerc...
1,imdb_4707,Acting,negative,Brian Keith's typical low-key style doesn't wo...
2,imdb_4707,Acting,negative,Elisha Cook Jr. goes over the top as a wild-ey...
3,imdb_4707,Movie quality,negative,a cheap-jack production all the way.
4,imdb_4707,Acting,positive,Beverly Garland who treats her role with chara...
5,imdb_16879,Emotional Impact,positive,gives them a relentless tug
6,imdb_16879,Plot,neutral,"witnesses the murder of a child, or does he?"
7,imdb_16879,Pacing,positive,here is no dialogue until the last few scenes
8,imdb_1907,Acting,negative,"this display of ""mega-coolness"""
9,imdb_1907,Plot,negative,"the movie's ""hero"" rises up and says something"


In [ ]:
print("CATEGORY DISTRIBUTION:")
print(results_df["category"].value_counts())

CATEGORY DISTRIBUTION:
category
Acting              13
Movie quality        9
Plot                 9
Comedy               4
Emotional Impact     4
Other                4
Recommendation       3
Directing            2
Cinematography       2
Pacing               1
Name: count, dtype: int64


In [ ]:
results_df.to_csv("aspect_results.csv", index=False)
print("✓ Saved: aspect_results.csv")

 # Pipeline 2 - Thematic Summarization

In [ ]:
class Theme(BaseModel):
    theme: str
    summary: str
    quote: str          # EXACT quote from source
    keywords: list[str]

THEME_PROMPT = """Extract themes from this text.

For each theme provide:
- theme: short name for the theme
- summary: 1-2 sentence summary
- quote: EXACT word-for-word quote from the text (copy directly)
- keywords: list of 3-5 relevant keywords

Return as JSON array. Quotes must be EXACT copies from the source text."""

class ExtractThemes(dspy.Signature):
    __doc__ = THEME_PROMPT

    text: str = dspy.InputField(desc="The text to analyze")
    themes: list[Theme] = dspy.OutputField(desc="List of themes")

theme_extractor = dspy.Predict(ExtractThemes)
print("✓ Theme extractor ready!")

✓ Theme extractor ready!


In [ ]:
result = theme_extractor(text=df_themes.iloc[50]['text'])

for t in result.themes:
    print(f"\nTheme: {t.theme}")
    print(f"Summary: {t.summary}")
    print(f"Quote: \"{t.quote}\"")
    print(f"Keywords: {t.keywords}")


Theme: Fire of the Chaldeans
Summary: The passage discusses Abraham’s encounter with the Chaldean worship of fire, portraying him as refusing idolatry and escaping divine protection.
Quote: "And Abraham was put into the fire because he refused to worship fire, which the Chaldeans honour; and that he escaped through God’s help, and fled from the fire of idolatry."
Keywords: ['fire', 'idolatry', 'Chaldeans', 'Abraham', 'divine protection']

Theme: Abraham’s Family and Marriages
Summary: Details the wives of Abram and Nachor, naming Sarai and Melcha, and noting family relations among early humanity.
Quote: "The name of Abram’s wife was Sarai, and the name of Nachor’s wife was Melcha, the daughter of Aran."
Keywords: ['family', 'marriage', 'Sarai', 'Melcha', 'Abram']

Theme: Chronology of Abraham
Summary: Highlights the age of Abram at departure from Charra and raises questions about the timeline with Thara’s death.
Quote: "And Abram was 75 years old when he went out from Charra."
Keyword

In [ ]:
THEME_INSTRUCTIONS = """
Extract main themes. For each:
- Theme name
- Brief summary
- EXACT quote (word-for-word from text)
- Keywords
"""

df_themes["prompt_preview"] = df_themes["text"].apply(
    lambda x: f"{THEME_INSTRUCTIONS}\n\nTEXT:\n{x[:400]}..."
)
df_themes[["id", "prompt_preview"]].head(5)

,id,prompt_preview
0,chunk_016,\nExtract main themes. For each:\n- Theme name...
1,chunk_231,\nExtract main themes. For each:\n- Theme name...
2,chunk_245,\nExtract main themes. For each:\n- Theme name...
3,chunk_272,\nExtract main themes. For each:\n- Theme name...
4,chunk_145,\nExtract main themes. For each:\n- Theme name...


In [ ]:
theme_results = []

for idx, row in tqdm(df_themes.head().iterrows(), total=20):
    try:
        result = theme_extractor(text=row["text"])
        for t in result.themes:
            theme_results.append({
                "chunk_id": row["id"],
                "source_text": row["text"],
                "theme": t.theme,
                "summary": t.summary,
                "quote": t.quote,
                "keywords": ", ".join(t.keywords)
            })
    except Exception as e:
        print(f"Error {row['id']}: {e}")

print(f"\n✓ Found {len(theme_results)} themes")

 20%|██        | 4/20 [04:14<18:51, 70.74s/it]

Error chunk_272: 'NoneType' object is not iterable


 25%|██▌       | 5/20 [05:06<15:20, 61.35s/it]


✓ Found 16 themes


In [ ]:
# After a call, inspect history
print(lm.history[-1])

In [ ]:
themes_df = pd.DataFrame(theme_results)
themes_df.head(50)

,chunk_id,source_text,theme,summary,quote,keywords
0,chunk_016,Introduction\n9\nBut as Kamesar rightly object...,Jerome's Hebrew Preference,Jerome consistently favors the Hebrew text ove...,Never in this book does Jerome prefer LXX to H...,"Hebrew, LXX, preference, correction, Jerome"
1,chunk_016,Introduction\n9\nBut as Kamesar rightly object...,QHG as a Definitive Work,"QHG is portrayed as a definitive, non‑transiti...",There is nothing transitional or experimental ...,"definitive, transitional, experimental, Hebrew..."
2,chunk_016,Introduction\n9\nBut as Kamesar rightly object...,Alternative Views on QHG,"Contrasting scholarly opinions, Cavallera and ...","Yet Cavallera, somewhat in the manner of Schad...","Cavallera, Schade, transitional, QHG, scholarl..."
3,chunk_231,125) prefigured Christ’s passion and the purpl...,Hebrew Lexicon,Analysis of the Hebrew term masqêh reveals its...,The Hebrew word under consideration here is ma...,"Hebrew, masqêh, butler, lexicon"
4,chunk_231,125) prefigured Christ’s passion and the purpl...,Jerome's Translation,"Jerome's renderings, such as translating a Heb...",Jerome translated it as ‘flour’ in Vg:,"Jerome, translation, flour, Vg"
5,chunk_231,125) prefigured Christ’s passion and the purpl...,Cupbearer Myth,The commentary draws a parallel between the bi...,"Ganymede, son of Tros according to Homer, Ilia...","Ganymede, cupbearer, mythology, Olympus"
6,chunk_231,125) prefigured Christ’s passion and the purpl...,LXX and Targum Variants,The passage shows how the LXX and various Targ...,The variety of meanings which he proposes for ...,"LXX, Targums, translation, variation"
7,chunk_231,125) prefigured Christ’s passion and the purpl...,Trinitarian Allusion,Didymus the Blind's reference to three shoots ...,"Didymus the Blind, De Trin. 1. 18, referred th...","Trinity, Didymus, shoots, symbolism"
8,chunk_245,"Commentary\nthese Patriarchs as Pharisees, and...",Biblical Symbolism as Christ,Jerome interprets the ‘tender shoot’ in the LX...,who takes the ‘tender shoot’ of LXX as a type ...,"tender shoot, Christ, typology, resurrection, LXX"
9,chunk_245,"Commentary\nthese Patriarchs as Pharisees, and...",Jewish Scribe Tradition,The text notes that descendants of Simeon and ...,the descendants of Simeon and Levi were scribe...,"Simeon, Levi, scribes, teachers, Torah"


In [ ]:
def find_best_match(quote, source, threshold=90):
    """Find best matching substring using fuzzy matching."""
    q = quote.lower().strip()
    s = source.lower()

    # Exact match
    if q in s:
        return True, 100, quote

    # Sliding window fuzzy match
    best_ratio, best_match = 0, ""
    q_len = len(q)

    for win in [q_len, int(q_len*0.8), int(q_len*1.2)]:
        if win <= 0: continue
        for i in range(0, max(1, len(s)-win+1), 15):
            window = s[i:i+win]
            ratio = fuzz.ratio(q, window)
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = source[i:i+win]

    return best_ratio >= threshold, best_ratio, best_match


def validate_quotes(df, threshold=90):
    """Validate all quotes against source text."""
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        valid, ratio, matched = find_best_match(row["quote"], row["source_text"], threshold)
        results.append({"quote_valid": valid, "match_ratio": ratio, "matched_text": matched})

    for col in ["quote_valid", "match_ratio", "matched_text"]:
        df[col] = [r[col] for r in results]
    return df

print("✓ Validation ready!")

✓ Validation ready!


In [ ]:
# ============================================================
MATCH_THRESHOLD = 90  # Minimum similarity %
# ============================================================

themes_df = validate_quotes(themes_df, threshold=MATCH_THRESHOLD)

# Show invalid quotes
invalid = themes_df[~themes_df["quote_valid"]]

if len(invalid) > 0:
    print(f"INVALID QUOTES ({len(invalid)}):")
    for _, row in invalid.head(5).iterrows():
        print(f"\n[{row['chunk_id']}] {row['match_ratio']}%")
        print(f"LLM:   \"{row['quote'][:80]}...\"")
        print(f"Found: \"{row['matched_text'][:80]}...\"")
else:
    print("✓ All quotes valid!")

100%|██████████| 16/16 [00:00<00:00, 345.36it/s]

✓ All quotes valid!


In [ ]:
# All results
themes_df[["chunk_id", "theme", "summary", "quote", "keywords",
           "quote_valid", "match_ratio"]].to_csv("theme_results.csv", index=False)
print("✓ Saved: theme_results.csv")

✓ Saved: theme_results.csv
